# Titanic 本番

**モード:** 本番（`competition/`）＝ **協力戦**。AI も道具として駆使して、実際にリーダーボードを上げにいく。
方向（何を試すか・いつ提出するか）は自分が決め、コードは Claude と一緒に書いて回す。

**評価指標:** accuracy（正解率）

**これまでの歩み（手元CV → public）:**
| 段階 | 正直CV | public |
|---|---|---|
| 最初のモデル（過学習） | 0.833 | 0.758 |
| gap改善＋アンサンブル | 0.839 | 0.763 |
| **＋家族生死 FamilySurv** | **0.848** | **0.780** 🏆 |
| （gender ベースの壁） | — | 0.766 → **突破** |

**効いた考え方:** ①正直な物差し(seed複数平均CV)を先に作る → ②過学習gapを縮める → ③本当に新しい情報(家族生死)を足す。
手元CVが上がるたび public も素直に上がった＝物差しが信用できた。

本番の流れ:
```
train.csv (Survived有) ──fit──▶ モデル ──predict──▶ test.csv (Survived無)
                                                        │
                                                        ▼
                                  submissions/submission.csv  (PassengerId, Survived)
                                                        │  kaggle competitions submit
                                                        ▼
                                                  リーダーボード
```
データ: `data/train.csv`(891) / `data/test.csv`(418) ／ 提出は `submissions/submission.csv`（2列・`index=False`）

## 📋 使うデータ：タイタニック号の「乗客名簿」

1912年に氷山にぶつかって沈んだ豪華客船タイタニック号。これはその **実際に乗っていた人たちの記録**。
やること = **乗客のプロフィール（年齢・性別・部屋ランク…）から「生き残ったか(1) / 亡くなったか(0)」を当てる。**
（"お客のプロフィールから退会する/しないを当てる" のと構造は同じ。当てる対象が今回は生死）

### 2つのファイル（ここが本番の肝）
| ファイル | 人数 | Survived(正解) | 役割 |
|---|---|---|---|
| `train.csv` | 891 | **あり** | 「どんな人が助かったか」を勉強させる過去問 |
| `test.csv`  | 418 | **なし** | この人たちの生死を予測して Kaggle に提出する本番問題 |

### 列の意味（1人目 = `Braund, Mr. Owen Harris` を例に）
| 列 | 意味 | 1人目の値 |
|---|---|---|
| **Survived** | 🎯 **正解（当てたいもの）** 1=生存 / 0=死亡 | 0（死亡） |
| Pclass | 部屋ランク　1=上等 / 2=中 / 3=下 | 3 |
| Name | 名前（敬称 Mr / Miss / Mrs が手がかり） | Braund, Mr. … |
| Sex | 性別 | male |
| Age | 年齢 | 22 |
| SibSp | 同乗した「兄弟・配偶者」の数 | 1 |
| Parch | 同乗した「親・子」の数 | 0 |
| Ticket | チケット番号 | A/5 21171 |
| Fare | 払った運賃 | 7.25 |
| Cabin | 部屋番号（**8割が欠損**） | (欠損) |
| Embarked | 乗船した港 S / C / Q | S |

> 生死はランダムじゃなく **かたより** がある（例: 女性ほど・上等な部屋ほど助かりやすい）。
> このかたより = 当てるための「ヒント」。次のセルで実際に生存率を見て確かめる。

## 1. 読み込み・EDA

In [1]:
# ============================================================
# 節1: データを読み込んで「どんなデータか」をつかむ（EDA）
#   狙い = 節2の前処理を決めるための材料集め
#   見るのは4点: 形 / 中身 / 欠損 / 目的変数の偏り
# ============================================================
import pandas as pd

# train = 答え(Survived)つきの学習用 / test = 答えなし・これを予測して提出する
train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")

# 1) 形: 行数(人数)と列数。test は Survived が無いぶん1列少ない = これが本番の構造
print("train:", train.shape, "  test:", test.shape)

# 2) 中身: 実際の値を数行眺める（どんな列があるか掴む）
display(train.head())

# 3) 欠損: 列ごとの「空っぽ」の数を多い順に ← 節2で「埋める/捨てる」を決める根拠
print("--- train 欠損数（多い順）---")
print(train.isnull().sum().sort_values(ascending=False))
print("\n--- test 欠損数（多い順）---")
print(test.isnull().sum().sort_values(ascending=False))

# 4) 目的変数の偏り: Survived の 0/1 割合
#    「全員を多数派(0)と予測」しただけで取れる点 = 何もしないベースラインの感覚
print("\n--- Survived の割合（0=死亡 / 1=生存）---")
print(train["Survived"].value_counts(normalize=True).round(3))

train: (891, 12)   test: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


--- train 欠損数（多い順）---
Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

--- test 欠損数（多い順）---
Cabin          327
Age             86
Fare             1
Name             0
Pclass           0
PassengerId      0
Sex              0
Parch            0
SibSp            0
Ticket           0
Embarked         0
dtype: int64

--- Survived の割合（0=死亡 / 1=生存）---
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64


In [2]:
# ============================================================
# 節1.5: 「その列、本当に生死のヒント？」を生存率で確かめる（EDA）
#   なぜ: 節2で特徴量にする前に「生死と差が出る列か」を先に確認する。
#         差が大きい列ほど強いヒント = モデルが頼る手がかり。
#   見方: groupby(列)["Survived"].mean() = その仲間の生存率（平均）
#         全体の生存率(=基準線)より上/下に大きくズレてるほど「効くヒント」
# ============================================================

# 名前から敬称(Title)をその場で取り出す（Mr/Miss/Mrs… 生死との関係を見るため）
title = train["Name"].str.extract(r",\s*([^.]+)\.")[0]

print("◆ 全体の生存率(基準線) =", round(train["Survived"].mean(), 3), "  ← 各グループはこれより上か下か？")

print("\n--- Sex（性別）ごとの生存率 ---")
print(train.groupby("Sex")["Survived"].mean().round(3))

print("\n--- Pclass（部屋ランク）ごとの生存率 ---")
print(train.groupby("Pclass")["Survived"].mean().round(3))

print("\n--- Title（敬称）ごとの生存率と人数 ---")
print(train.groupby(title)["Survived"].agg(["mean", "count"]).round(3)
      .sort_values("count", ascending=False).head(6))

print("\n--- SibSp（同乗の兄弟・配偶者の数）ごとの生存率 ---")
print(train.groupby("SibSp")["Survived"].mean().round(3))

◆ 全体の生存率(基準線) = 0.384   ← 各グループはこれより上か下か？

--- Sex（性別）ごとの生存率 ---
Sex
female    0.742
male      0.189
Name: Survived, dtype: float64

--- Pclass（部屋ランク）ごとの生存率 ---
Pclass
1    0.630
2    0.473
3    0.242
Name: Survived, dtype: float64

--- Title（敬称）ごとの生存率と人数 ---
         mean  count
0                   
Mr      0.157    517
Miss    0.698    182
Mrs     0.792    125
Master  0.575     40
Dr      0.429      7
Rev     0.000      6

--- SibSp（同乗の兄弟・配偶者の数）ごとの生存率 ---
SibSp
0    0.345
1    0.536
2    0.464
3    0.250
4    0.167
5    0.000
8    0.000
Name: Survived, dtype: float64


## 2. 前処理

In [3]:
# ============================================================
# 節2: 前処理 + 特徴量づくり（最適化後の確定版 = 10特徴）
#   方針: 特徴量は「増やせば良い」ではない。探索の結果、効いたのは
#         Title / Deck / HasCabin だけ。年齢ビン・運賃ビン等は不採用(小データで焼き直しは逆効果)。
#   リーク回避: train+test を縦結合して『特徴量だけ』から作る(正解Survivedは一切使わない)。
# ============================================================
ntr  = len(train)
full = pd.concat([train, test], sort=False, ignore_index=True)   # test の Survived は自動で NaN

# 敬称(Title): 名前から抜き、珍しいものは Rare に集約
title_raw = full["Name"].str.extract(r",\s*([^.]+)\.")[0].str.strip()
full["Title"] = title_raw.map({"Mr":"Mr","Miss":"Miss","Mrs":"Mrs","Master":"Master",
                               "Mlle":"Miss","Ms":"Miss","Mme":"Mrs"}).fillna("Rare")

# 欠損補完（中央値はグループ別: 年齢=敬称ごと / 運賃=Pclassごと）
full["Age"]      = full["Age"].fillna(full.groupby("Title")["Age"].transform("median"))
full["Fare"]     = full["Fare"].fillna(full.groupby("Pclass")["Fare"].transform("median"))
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])

# 家族系
full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
full["IsAlone"]    = (full["FamilySize"] == 1).astype(int)

# 部屋(Cabin)
full["Deck"]     = full["Cabin"].str[0].fillna("U")   # 頭文字(C123→C)、欠損→U
full["HasCabin"] = full["Cabin"].notna().astype(int)  # 部屋番号がある=上等客の手がかり

# 文字 → 数値
full["Sex"]      = full["Sex"].map({"male":0, "female":1})
full["Embarked"] = full["Embarked"].map({"S":0, "C":1, "Q":2})
full["Title"]    = full["Title"].map({"Mr":0, "Miss":1, "Mrs":2, "Master":3, "Rare":4})
decks            = ["U","A","B","C","D","E","F","G","T"]
full["Deck"]     = full["Deck"].map({d:i for i,d in enumerate(decks)}).astype(int)

# train/test に戻す（Name/Ticket/Fare は節2.5の家族生死で使うので残す）
train_p = full.iloc[:ntr].reset_index(drop=True)
test_p  = full.iloc[ntr:].reset_index(drop=True)

features = ["Pclass","Sex","Age","Fare","Embarked","FamilySize","IsAlone","Title","Deck","HasCabin"]
y = train_p["Survived"].astype(int)

print("基本特徴量(10):", features)
print("train_p:", train_p.shape, " test_p:", test_p.shape)
print("欠損残り  train:", int(train_p[features].isnull().sum().sum()),
      " / test:", int(test_p[features].isnull().sum().sum()))
display(train_p[features].head())

基本特徴量(10): ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone', 'Title', 'Deck', 'HasCabin']
train_p: (891, 17)  test_p: (418, 17)
欠損残り  train: 0  / test: 0


,Pclass,Sex,Age,Fare,Embarked,FamilySize,IsAlone,Title,Deck,HasCabin
0,3,0,22.0,7.2500,0,2,0,0,0,0
1,1,1,38.0,71.2833,1,2,0,2,3,1
2,3,1,26.0,7.9250,0,1,1,1,0,0
3,1,1,35.0,53.1000,0,2,0,2,3,1
4,3,0,35.0,8.0500,0,1,1,0,0,0


## 2.5 家族・グループの生死 `FamilySurv` — gender の壁を破る一手

**問題:** 上の10特徴は、ぜんぶ「**その人ひとりの属性**」（性別・年齢・部屋…）。
でも生死は **家族・グループ単位で揃う**（同じ救命ボートに乗れたか / 乗れなかったか）。
だから「女性は生存」「男性は死亡」の**例外**を当てられない:

- 家族が**全滅**した女性 → 本人も**死にやすい**（gender ルールの例外）
- 家族が**生還**した男性 → 本人も**生きやすい**（gender ルールの例外）

**解決:** 「同じ苗字＋運賃」「同じチケット」の人を同一グループとみなし、
そのグループの**他のメンバーの生死**を特徴量にする（1=誰か生存 / 0=誰か死亡 / 0.5=情報なし）。

**⚠️ リーク厳禁:** ある人の値は **「自分以外 ＆ 答えを知っている人(known)」** の生死だけから作る。
検証foldの正解を覗くと、CVだけ上がって本番で崩れる（今まで避けてきた罠の最悪版）。
→ だから下の関数は `known_mask` を受け取り、CVでは**毎foldで学習foldの正解だけ**から作り直す。

In [4]:
# ============================================================
# 節2.5: 家族生死 FamilySurv を「リークなし」で作る関数
# ============================================================
import numpy as np

def family_survival(df, known_mask, survived):
    """各行に 0.5(中立)/1.0(家族の誰か生存)/0.0(家族の誰か死亡) を付ける。
    known_mask[i]=True は『その人の Survived を見てよい(=学習fold/既知)』。
    others を "自分以外 ＆ known" に限定 → 自分の答えも検証foldの答えも漏れない。"""
    fs   = pd.Series(0.5, index=df.index, dtype=float)
    last = df["Name"].str.split(",").str[0].str.strip()        # 苗字

    def apply_group(keys, only_unset):
        for _, idx in df.groupby(keys).groups.items():
            idx = list(idx)
            if len(idx) <= 1:                  # 一人だけのグループは手がかりなし
                continue
            for i in idx:
                if only_unset and fs[i] != 0.5:
                    continue
                others = [j for j in idx if j != i and known_mask[j]]   # 自分以外 ＆ 既知
                if not others:
                    continue
                s = survived.loc[others]
                if   s.max() == 1: fs[i] = 1.0     # 家族の誰かが生存
                elif s.min() == 0: fs[i] = 0.0     # 家族の誰かが死亡
    apply_group([last, df["Fare"]], only_unset=False)   # ① 苗字+運賃 = 家族
    apply_group(df["Ticket"],       only_unset=True)    # ② 同チケット = 同行(①で埋まらなかった人を補う)
    return fs

# 提出/体温計で使う「全trainを既知」とした値を一度だけ作る（test には train の家族から付く）
FULL  = pd.concat([train_p, test_p], ignore_index=True)
KNOWN = np.zeros(len(FULL), dtype=bool); KNOWN[:len(train_p)] = True
SURV  = pd.Series(np.where(KNOWN, FULL["Survived"].values, np.nan), index=FULL.index)
_fs_all      = family_survival(FULL, KNOWN, SURV)
FS_TRAIN_ALL = _fs_all.iloc[:len(train_p)].values   # train 用（各自 他のtrain家族を参照）
FS_TEST_ALL  = _fs_all.iloc[len(train_p):].values   # test 用（train の家族を参照）

print("FamilySurv 分布(train):", dict(pd.Series(FS_TRAIN_ALL).value_counts()))
print("FamilySurv 分布(test) :", dict(pd.Series(FS_TEST_ALL).value_counts()))
print("→ 約4割に家族の手がかりが付く（残り0.5は従来通り中立）")

FamilySurv 分布(train):

 {0.5: np.int64(516), 1.0: np.int64(199), 0.0: np.int64(176)}
FamilySurv 分布(test) : {0.5: np.int64(247), 1.0: np.int64(96), 0.0: np.int64(75)}
→ 約4割に家族の手がかりが付く（残り0.5は従来通り中立）


## 3. モデル（LightGBM 単体）＋ 正直な手元CV

In [30]:
# ============================================================
# 節3: モデル(LightGBM 単体) + 正直な手元CV
#   モデル: チューニング済み LightGBM 1個（シンプル重視）。
#           アンサンブルより本番は気持ち下がるかもだが差は誤差レベル → 読みやすさを優先。
#   CV   : seedを複数変えた平均 = まぐれ(当たりseed)に騙されない物差し。
#          FamilySurv は毎foldで『学習foldの正解だけ』から作り直す → リークなし。
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

# ランダムサーチ(250通り)で見つかった LightGBM の設定
LGB_PARAMS = dict(num_leaves=11, min_child_samples=18, n_estimators=508,
                  learning_rate=0.024006646754962464, reg_alpha=0.6906159513177683,
                  reg_lambda=2.6656926996178574, subsample=0.9663023921374498,
                  colsample_bytree=0.8296155855603391, max_depth=6,
                  subsample_freq=1, random_state=42, verbose=-1, n_jobs=1)

def make_model():
    """チューニング済み LightGBM 単体"""
    return lgb.LGBMClassifier(**LGB_PARAMS)

FEATS_FS = features + ["FamilySurv"]   # 10特徴 + 家族生死 = 11

def honest_cv(seeds=range(5), n_splits=5):
    """seedを変えて 5-fold を回す。各foldで FamilySurv を学習foldの正解だけから作り直す。"""
    sc = []
    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tri, vai in skf.split(train_p, y):
            known = np.zeros(len(train_p), dtype=bool); known[tri] = True
            surv  = pd.Series(np.where(known, y.values, np.nan), index=train_p.index)
            fs    = family_survival(train_p, known, surv)            # 学習foldの正解だけで作る
            Xtr = train_p.iloc[tri][features].copy(); Xtr["FamilySurv"] = fs.iloc[tri].values
            Xva = train_p.iloc[vai][features].copy(); Xva["FamilySurv"] = fs.iloc[vai].values
            m = make_model().fit(Xtr, y.iloc[tri])
            sc.append((m.predict(Xva) == y.iloc[vai].values).mean())
    return np.array(sc)

scores = honest_cv(seeds=range(5))   # 5seed×5fold=25回
print("正直CV accuracy: %.4f ± %.4f" % (scores.mean(), scores.std()))

# --- 過学習の体温計（FamilySurvは全trainから=FS_TRAIN_ALL を使用）---
X = train_p[features].copy(); X["FamilySurv"] = FS_TRAIN_ALL
m_full = make_model().fit(X, y)
train_acc = (m_full.predict(X) == y).mean()
print("\n--- 過学習の体温計 ---")
print("丸暗記(train): %.4f" % train_acc)
print("初見  (CV)   : %.4f" % scores.mean())
print("gap          : %.4f   ← 小さいほど本番で落ちにくい" % (train_acc - scores.mean()))

正直CV accuracy: 0.8438 ± 0.0256

--- 過学習の体温計 ---
丸暗記(train): 0.9057
初見  (CV)   : 0.8438
gap          : 0.0619   ← 小さいほど本番で落ちにくい


## 4. test を予測 → submission.csv（PassengerId, Survived / index=False）

In [6]:
# ============================================================
# 節4: test を予測して提出ファイルを作る
#   家族生死は「全trainを既知」として作った値(FS_TRAIN_ALL / FS_TEST_ALL)を使う。
#   モデルは全trainで学習し直す(CVより多いデータで本番モデルを作る)。
# ============================================================
X      = train_p[features].copy(); X["FamilySurv"]      = FS_TRAIN_ALL
X_test = test_p[features].copy();  X_test["FamilySurv"] = FS_TEST_ALL

model = make_model().fit(X, y)            # 11特徴・LightGBM単体で本番モデル
pred  = model.predict(X_test).astype(int) # 答えを知らない test 418人を予測

submission = pd.DataFrame({
    "PassengerId": test_p["PassengerId"].astype(int),
    "Survived": pred,
})
submission.to_csv("submissions/submission.csv", index=False)   # index=False=行番号を書かない

print("submission:", submission.shape)
print(submission["Survived"].value_counts())
display(submission.head())

submission: (418, 2)
Survived
0    267
1    151
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## 5. 提出（自分で）

```
kaggle competitions submit -c titanic -f submissions/submission.csv -m "メッセージ"
```
提出したら public スコアを持ってきて。予想と照らして「なぜその点か」を一緒に開ける。